# 主動型投資- 資料爬蟲
**以「週」為單位做台股的買賣，賺價差**  
數據(yahoo finance上個股的資訊)：
*  日期(每周最後交易日)
*  股價(每週最後交易日當天收盤價)
*  大戶(三大法人)買賣超張數
*  大戶籌碼比例
*  借券賣出餘額樟張數
*  融資餘額張數
*  券資比
*  融資維持率




## 取得資料
數據來源url (ABBR為台股代碼；若為上櫃/興櫃股票，url中的TW要改為TWO)：
* 法人買賣超：https://tw.finance.yahoo.com/quote/ABBR.TW/institutional-trading (要選則「週」)
* 資券變化：https://tw.finance.yahoo.com/quote/ABBR.TW/margin (要選擇「資券餘額/借券賣出餘額」&「週」)
* 大戶籌碼：https://tw.finance.yahoo.com/quote/ABBR.TW/major-holders
* 股價(週K線)：https://tw.stock.yahoo.com/quote/ABBR.TW/technical-analysis (要選擇「週」；從 K 線圖上方的資訊列讀取 日期/開/高/低/收)

### 匯入Package

In [1]:
import asyncio
import re
import sys
import threading
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from statistics import median

import pandas as pd
from playwright.async_api import async_playwright

### 爬蟲函式定義(共用、法人買賣超、資券餘額/借券賣出餘額、大戶籌碼、股價週K線)

In [2]:
# ===== 共用：變數與函式 =====
# (輸入代碼、開網頁、點按鈕、確認週資料、讀表格、跑 Playwright、存檔；下方各資料的爬蟲都會用到)

URL_TEMPLATE = "https://tw.finance.yahoo.com/quote/{symbol}/{page}"
SUFFIXES = (".TW", ".TWO")  # 上市用 .TW；上櫃、興櫃用 .TWO
SELECTED_BG = "rgb(218, 237, 255)"  # 按鈕被選取時的淡藍底色 (#daedff)
PERIOD_BUTTONS = ("日", "週", "月", "季")
STALE_RETRIES = 3  # 週資料是 Yahoo 舊快取時，最多重新載入頁面幾次
STALE_RETRY_WAIT_S = 10  # 每次重新載入前等幾秒
DATA_DIR = "DataCollection"  # 各股票資料夾 (ABBR) 的上層資料夾 (相對於 notebook 所在的資料夾)


def ask_stock_code():
    """請使用者輸入 4 位數字的台股代碼"""
    while True:
        abbr = input("請輸入台股代碼(4位數字)：").strip()
        if re.fullmatch(r"\d{4}", abbr):
            return abbr
        print(f"「{abbr}」不是 4 位數字，請重新輸入。")


def looks_weekly(dates):
    """用表格日期判斷是否為「週」資料：每列落在不同週，且相鄰兩列間隔的中位數約 7 天"""
    if len(dates) < 3:
        return False
    gaps = [(a - b).days for a, b in zip(dates, dates[1:])]
    iso_weeks = [d.isocalendar()[:2] for d in dates]
    return all(g > 0 for g in gaps) and len(set(iso_weeks)) == len(iso_weeks) and 5 <= median(gaps) <= 9


def looks_daily(dates):
    """用表格日期判斷是否為「日」資料：日期由新到舊，相鄰兩列間隔的中位數不超過 3 天 (週末會隔 3 天)"""
    if len(dates) < 3:
        return False
    gaps = [(a - b).days for a, b in zip(dates, dates[1:])]
    return all(g > 0 for g in gaps) and median(gaps) <= 3


class StaleWeekDataError(RuntimeError):
    """Yahoo 偶爾會回傳舊的週資料快取 (例如 11/10 執行，拿到的週資料只算到 10/27)"""


async def _launch_browser(p, headless):
    # 優先使用 Windows 內建的 Edge，失敗再用 Playwright 自帶的 Chromium (需先執行 playwright install chromium)
    try:
        return await p.chromium.launch(channel="msedge", headless=headless)
    except Exception:
        return await p.chromium.launch(headless=headless)


async def _open_section(page, abbr, page_name, heading, timeout_s):
    """依序嘗試 .TW、.TWO 網址，回傳 (實際代號, 標題含 heading 且有表格的區塊)
    page_name 是網址最後一段，例如 institutional-trading、margin、major-holders"""
    tried = []
    for suffix in SUFFIXES:
        symbol = abbr + suffix
        url = URL_TEMPLATE.format(symbol=symbol, page=page_name)
        tried.append(url)
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout_s * 1000)

        # Yahoo 有時會自動導正代號，以頁面標題上的代號為準，例如「環球晶(6488.TWO) 法人買賣」
        m = re.search(rf"\(({abbr}\.TWO?)\)", await page.title())
        if m is None or m.group(1) != symbol:
            continue
        # 沒有這項資料的股票(例如興櫃)會被導回報價頁
        if f"/{page_name}" not in page.url:
            continue

        # 同一頁可能有好幾個標題含 heading 的區塊 (例如大戶籌碼頁)，只取有表格的那個
        section = page.locator("section", has=page.locator("h2", has_text=heading)).filter(
            has=page.locator("li .table-row")
        )
        try:
            await section.wait_for(timeout=timeout_s * 1000)
            return symbol, section
        except Exception:
            continue
    raise RuntimeError(
        f"找不到代碼 {abbr} 的「{heading}」資料(興櫃股票 Yahoo 沒有提供)。已嘗試：\n" + "\n".join(tried)
    )


async def _open_section_fresh(page, abbr, page_name, heading, timeout_s, read_fn):
    """開頁面後執行 read_fn(section) 讀資料，回傳 (實際代號, read_fn 的結果)
    週資料是 Yahoo 的舊快取 (StaleWeekDataError) 就等一下、重新載入頁面再讀"""
    for attempt in range(1, STALE_RETRIES + 1):
        symbol, section = await _open_section(page, abbr, page_name, heading, timeout_s)
        try:
            return symbol, await read_fn(section)
        except StaleWeekDataError as e:
            if attempt == STALE_RETRIES:
                raise StaleWeekDataError(
                    f"{symbol}「{heading}」{e}，重試 {STALE_RETRIES} 次仍是舊資料。"
                    "Yahoo 的週資料可能還沒更新，請過一段時間再執行 (舊的 csv 不會被刪除)。"
                ) from None
            print(f"{symbol}「{heading}」{e}，可能是 Yahoo 的舊快取，{STALE_RETRY_WAIT_S} 秒後重新載入 ({attempt}/{STALE_RETRIES - 1})")
            await asyncio.sleep(STALE_RETRY_WAIT_S)


async def _click_until_selected(page, section, name, group, timeout_s):
    """點擊按鈕 name，直到它的背景變成淡藍色，且同一組 (group) 的其他按鈕都不是淡藍色
    (頁面剛載入時點擊可能沒反應，所以沒變色就重新點)"""
    btn = section.get_by_role("button", name=name, exact=True)
    deadline = time.monotonic() + timeout_s
    while True:
        await btn.click()
        # 按鈕 hover 時也會變淡藍色，先把滑鼠移開以免誤判
        await page.mouse.move(0, 0)
        await asyncio.sleep(1)
        bg = await section.evaluate(
            "sec => Object.fromEntries([...sec.querySelectorAll('button')]"
            ".map(b => [b.innerText.trim(), getComputedStyle(b).backgroundColor]))"
        )
        if bg.get(name) == SELECTED_BG and all(bg.get(o) != SELECTED_BG for o in group if o != name):
            return
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：「{name}」按鈕沒有變成選取狀態，按鈕底色：{bg}")


def _flatten_row(row):
    """有分組的欄位 (例如「融資」底下的 增減/餘額/使用率％) 讀進來會是 list，
    比照範本 CSV 在前面補一個空白的分組標題欄"""
    flat = []
    for cell in row:
        if isinstance(cell, list):
            flat += [""] + cell
        else:
            flat.append(cell)
    return flat


async def _wait_rows(section, n_cols, timeout_s, check, label):
    """讀取表格直到日期通過 check (looks_weekly / looks_daily)，回傳 (rows, dates)
    (表格可能延遲更新，所以輪詢到符合為止；只保留欄位數為 n_cols 的列)"""
    deadline = time.monotonic() + timeout_s
    while True:
        rows = await section.evaluate(
            "sec => [...sec.querySelectorAll('li .table-row')].map(r => [...r.children].map(c =>"
            " c.querySelector('li') ? [...c.querySelectorAll('li')].map(l => l.innerText.trim())"
            " : c.innerText.trim()))"
        )
        rows = [_flatten_row(r) for r in rows]
        rows = [r for r in rows if len(r) == n_cols]
        dates = [datetime.strptime(r[0], "%Y/%m/%d").date() for r in rows]
        if check(dates):
            return rows, dates
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：表格日期看起來不是「{label}」資料，請重新執行。")
        await asyncio.sleep(0.5)


async def _wait_weekly_rows(section, n_cols, timeout_s):
    """讀取表格直到日期呈現「每週一筆」，回傳 (rows, dates)"""
    return await _wait_rows(section, n_cols, timeout_s, looks_weekly, "週")


def check_week_fresh(week_dates, latest_day):
    """週資料最新一列要和「日」資料的最新交易日落在同一週，否則視為 Yahoo 回傳的舊快取"""
    if week_dates[0].isocalendar()[:2] != latest_day.isocalendar()[:2]:
        raise StaleWeekDataError(f"週資料最新一列是 {week_dates[0]}，但日資料最新交易日是 {latest_day}")


async def _read_week_checked(page, section, n_cols, timeout_s):
    """先點「日」記下最新交易日，再點「週」讀週資料並確認不是舊快取，回傳 (rows, dates)"""
    await _click_until_selected(page, section, "日", PERIOD_BUTTONS, timeout_s)
    _, daily_dates = await _wait_rows(section, n_cols, timeout_s, looks_daily, "日")
    # 確認方法 1：「週」按鈕背景變成淡藍色，日/月/季按鈕不是淡藍色
    await _click_until_selected(page, section, "週", PERIOD_BUTTONS, timeout_s)
    # 確認方法 2：表格日期呈現「每週一筆」
    rows, dates = await _wait_weekly_rows(section, n_cols, timeout_s)
    # 確認方法 3：最新一週包含最新交易日 (Yahoo 偶爾會回傳好幾週、甚至好幾個月前的週資料快取)
    check_week_fresh(dates, daily_dates[0])
    return rows, dates


def _to_week_df(rows, dates, columns, symbol):
    df = pd.DataFrame(rows, columns=columns)
    # 第一欄是日期，格式比照範本 (2026/10/8，不補 0)
    df[columns[0]] = [f"{d.year}/{d.month}/{d.day}" for d in dates]
    df.attrs["symbol"] = symbol  # 記下實際使用的代號 (例如 6488.TWO)，不會寫進 CSV
    return df


def run_in_own_loop(coro_fn, *args, **kwargs):
    """Jupyter 本身已有 event loop (Windows 上還是不支援子行程的 Selector loop)，
    所以另開一個執行緒、用自己的 event loop 跑 Playwright。"""
    result = {}

    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result["value"] = loop.run_until_complete(coro_fn(*args, **kwargs))
        except BaseException as e:
            result["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


def clear_week_csv(abbr, file_tags, base_dir=DATA_DIR):
    """刪除 DataCollection/ABBR 資料夾內舊的 ABBR_{file_tag}_YYYYMMDD.csv (每個 file_tag 不論日期都刪)，回傳刪掉的檔案"""
    folder = Path(base_dir) / abbr
    removed = []
    for file_tag in file_tags:
        for old in folder.glob(f"{abbr}_{file_tag}_*.csv"):
            old.unlink()
            removed.append(old)
    return removed


def save_week_csv(df, abbr, file_tag, base_dir=DATA_DIR):
    """存成 DataCollection/ABBR/ABBR_{file_tag}_YYYYMMDD.csv (UTF-8 BOM、CRLF 換行，比照範本 CSV)"""
    folder = Path(base_dir) / abbr
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{abbr}_{file_tag}_{date.today():%Y%m%d}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig", lineterminator="\r\n")
    return path


# ===== 法人買賣超（週）爬蟲：函式定義 =====
# 網址 .../institutional-trading，先讀「日」的最新交易日，再點「週」後讀「法人逐日買賣超」表格

INSTITUTIONAL_COLUMNS = ["日期", "外資(張)", "投信(張)", "自營商(張)", "合計(張)", "外資籌碼", "漲跌幅(%)", "成交量"]


async def _scrape_institutional_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")

            async def read(section):
                return await _read_week_checked(page, section, len(INSTITUTIONAL_COLUMNS), timeout_s)

            symbol, (rows, dates) = await _open_section_fresh(
                page, abbr, "institutional-trading", "法人逐日買賣超", timeout_s, read
            )
        finally:
            await browser.close()
    return _to_week_df(rows, dates, INSTITUTIONAL_COLUMNS, symbol)


def fetch_institutional_trading_week(abbr, headless=True, timeout_s=30):
    """取得「法人逐日買賣超」→「週」的表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_institutional_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 資券餘額 / 借券賣出餘額（週）爬蟲：函式定義 =====
# 網址 .../margin，在「資券餘額逐日增減」依序切到兩個分頁，各自先讀「日」的最新交易日、再點「週」後讀表格

# 欄位比照 Margin_Week_11cols.csv、Securities_Lending_and_Borrowing_Week_8cols.csv
# 「融資」「融券」「借券賣出」是網頁上的分組標題，在 CSV 中是空白欄
MARGIN_COLUMNS = ["日期", "融資", "增減", "餘額", "使用率％", "融券", "增減", "餘額", "使用率％", "券資比", "資券互抵"]
LENDING_COLUMNS = ["日期", "融券", "增減", "餘額", "限額", "借券賣出", "增減", "餘額"]
MARGIN_TABS = {"資券餘額": MARGIN_COLUMNS, "借券賣出餘額": LENDING_COLUMNS}  # 分頁名稱: 欄位


async def _scrape_margin_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")

            async def read(section):
                results = []
                for tab, columns in MARGIN_TABS.items():
                    # 分頁按鈕要變成淡藍色
                    await _click_until_selected(page, section, tab, MARGIN_TABS, timeout_s)
                    # 兩個分頁的欄位數不同 (11 / 8)，用欄位數篩選就不會讀到切換前的舊表格
                    results.append(await _read_week_checked(page, section, len(columns), timeout_s))
                return results

            symbol, results = await _open_section_fresh(page, abbr, "margin", "資券餘額逐日增減", timeout_s, read)
        finally:
            await browser.close()
    return tuple(
        _to_week_df(rows, dates, columns, symbol) for (rows, dates), columns in zip(results, MARGIN_TABS.values())
    )


def fetch_margin_week(abbr, headless=True, timeout_s=30):
    """取得「資券餘額逐日增減」→「週」的兩個分頁，回傳 (資券餘額 DataFrame, 借券賣出餘額 DataFrame)"""
    return run_in_own_loop(_scrape_margin_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 大戶籌碼（週）爬蟲：函式定義 =====
# 網址 .../major-holders，表格本來就是每週一筆，不用點按鈕，只用表格日期確認

MAJOR_HOLDERS_COLUMNS = ["年度/日期", "外資籌碼", "大戶籌碼", "董監持股", "股價"]  # 比照 Major_Holders_Week_5cols.csv


async def _scrape_major_holders_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")
            # 區塊標題是「台積電(2330) 大戶籌碼」這種格式，用「大戶籌碼」比對
            symbol, section = await _open_section(page, abbr, "major-holders", "大戶籌碼", timeout_s)
            rows, dates = await _wait_weekly_rows(section, len(MAJOR_HOLDERS_COLUMNS), timeout_s)
        finally:
            await browser.close()
    return _to_week_df(rows, dates, MAJOR_HOLDERS_COLUMNS, symbol)


def fetch_major_holders_week(abbr, headless=True, timeout_s=30):
    """取得「大戶籌碼」的週表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_major_holders_week, abbr, headless=headless, timeout_s=timeout_s)

In [3]:
# ===== 股價（週K線）爬蟲：函式定義 =====
# 網址 https://tw.stock.yahoo.com/quote/ABBR.TW(O)/technical-analysis，先讀「日」的最新交易日，再點「週」後，
# 用滑鼠逐一移到每一根 K 棒上，讀取 K 線圖上方資訊列顯示的 日期/開/高/低/收 (K 線圖約顯示最近 120 週)

PRICE_URL_TEMPLATE = "https://tw.stock.yahoo.com/quote/{symbol}/technical-analysis"
PRICE_PERIOD_BUTTONS = ("5分", "15分", "30分", "60分", "日", "週", "月")
PRICE_COLUMNS = ["日期", "開", "高", "低", "收"]
PRICE_FILE_TAG = "Stock_Price_Week"
DAILY_CANDLES = 5  # 讀「日」K 線時只 hover 最右邊幾根，用來取得最新交易日

CANDLE_SELECTOR = "g.highcharts-candlestick-series .highcharts-point"
OHLC_BAR_SELECTOR = ".ohlcList-tooltip"  # K 線圖上方的資訊列：日期 開 高 低 收 量(張) 漲跌

# 第 i 根 K 棒的中心 x、繪圖區中央的 y、繪圖區左右邊界 (每次重新計算，避免廣告載入造成版面位移)
_CANDLE_POS_JS = """i => {
  const plot = document.querySelector('.highcharts-plot-background').getBoundingClientRect();
  const r = document.querySelectorAll('g.highcharts-candlestick-series .highcharts-point')[i].getBoundingClientRect();
  return [r.x + r.width / 2, plot.y + plot.height / 2, plot.x, plot.x + plot.width];
}"""

# 資訊列的每一格是「標籤 + 數值」(例如 開 58.4)，只有日期那格沒有標籤
_READ_OHLC_JS = """bar => [...bar.children].map(d => d.children.length >= 2
  ? [d.children[0].innerText.trim(), d.children[1].innerText.trim()]
  : ['日期', d.innerText.trim()])"""

# 圖表本身的 K 棒數值，只用來核對滑鼠讀到的數值
_CHART_POINTS_JS = """() => {
  const c = window.Highcharts && Highcharts.charts.filter(Boolean)[0];
  return c ? c.series[0].points.map(p => [p.x, p.open, p.high, p.low, p.close]) : null;
}"""


async def _open_technical_page(page, abbr, timeout_s):
    """依序嘗試 .TW、.TWO 網址，回傳實際代號 (例如 5498.TWO)"""
    tried = []
    for suffix in SUFFIXES:
        symbol = abbr + suffix
        url = PRICE_URL_TEMPLATE.format(symbol=symbol)
        tried.append(url)
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout_s * 1000)
        # 上櫃股票用 .TW 網址也會顯示頁面，但標題上的代號是 .TWO，例如「凱崴(5498.TWO) 技術分析」
        m = re.search(rf"\(({abbr}\.TWO?)\)", await page.title())
        if m is None or m.group(1) != symbol or "/technical-analysis" not in page.url:
            continue
        try:
            await page.locator(CANDLE_SELECTOR).first.wait_for(timeout=timeout_s * 1000)
            return symbol
        except Exception:
            continue
    raise RuntimeError(f"找不到代碼 {abbr} 的技術分析 K 線圖。已嘗試：\n" + "\n".join(tried))


async def _read_ohlc_bar(page):
    return dict(await page.locator(OHLC_BAR_SELECTOR).evaluate(_READ_OHLC_JS))


async def _hover_candles(page, timeout_s, last_n=None):
    """滑鼠由左到右移到每一根 K 棒上 (有給 last_n 時只移到最右邊 last_n 根)，讀取資訊列，
    回傳 [{日期, 開, 高, 低, 收}, ...]"""
    await page.locator(".highcharts-plot-background").first.scroll_into_view_if_needed()
    n_candles = await page.locator(CANDLE_SELECTOR).count()
    start = 0 if last_n is None else max(0, n_candles - last_n)
    prev_date = (await _read_ohlc_bar(page))["日期"]  # 還沒 hover 時顯示最新一根
    records = []
    for i in range(start, n_candles):
        x, y, left, right = await page.evaluate(_CANDLE_POS_JS, i)
        if not left <= x <= right:  # 只露出一半、中心在繪圖區外的 K 棒無法 hover
            continue
        # 相鄰兩根 K 棒的日期一定不同：資訊列的日期變了，才代表已讀到這一根
        deadline = time.monotonic() + timeout_s
        wiggle = 0
        while True:
            await page.mouse.move(x, y + wiggle)
            bar = await _read_ohlc_bar(page)
            if bar["日期"] != prev_date:
                break
            if time.monotonic() > deadline:
                raise RuntimeError(f"等待逾時：滑鼠移到第 {i + 1} 根 K 棒後，資訊列沒有更新")
            wiggle = 1 - wiggle  # 在原地上下移動 1px，觸發 mousemove
            await asyncio.sleep(0.1)
        prev_date = bar["日期"]
        records.append({col: bar[col] for col in PRICE_COLUMNS})
    return records


async def _read_candles(page, period, check, timeout_s, last_n=None):
    """點週期按鈕 period，hover K 棒並讀取數值，回傳 (records, 日期由新到舊)"""
    # 確認方法 1：週期按鈕背景變成淡藍色，其他週期按鈕不是淡藍色
    await _click_until_selected(page, page.locator("body"), period, PRICE_PERIOD_BUTTONS, timeout_s)
    # 確認方法 2：K 棒的日期通過 check (looks_weekly / looks_daily)；圖表可能還在重繪，所以輪詢到符合為止
    deadline = time.monotonic() + timeout_s
    while True:
        await asyncio.sleep(1.5)
        records = await _hover_candles(page, timeout_s, last_n)
        dates = sorted({datetime.strptime(r["日期"], "%Y/%m/%d").date() for r in records}, reverse=True)
        if len(dates) == len(records) and check(dates):
            return records, dates
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：K 線圖的日期看起來不是「{period}」資料，請重新執行。")


async def _scrape_stock_price_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            # 視窗要夠高，整個 K 線圖才能捲到畫面內讓滑鼠 hover
            page = await browser.new_page(locale="zh-TW", viewport={"width": 1400, "height": 1200})
            # 週資料是 Yahoo 的舊快取 (StaleWeekDataError) 就等一下、重新載入頁面再讀
            for attempt in range(1, STALE_RETRIES + 1):
                symbol = await _open_technical_page(page, abbr, timeout_s)
                # 先點「日」記下最新交易日，再點「週」讀每一根 K 棒
                _, daily_dates = await _read_candles(page, "日", looks_daily, timeout_s, last_n=DAILY_CANDLES)
                records, dates = await _read_candles(page, "週", looks_weekly, timeout_s)
                try:
                    # 確認方法 3：最新一週包含最新交易日 (Yahoo 偶爾會回傳好幾週前的週資料快取)
                    check_week_fresh(dates, daily_dates[0])
                    break
                except StaleWeekDataError as e:
                    if attempt == STALE_RETRIES:
                        raise StaleWeekDataError(
                            f"{symbol} 週K線{e}，重試 {STALE_RETRIES} 次仍是舊資料。"
                            "Yahoo 的週資料可能還沒更新，請過一段時間再執行 (舊的 csv 不會被刪除)。"
                        ) from None
                    print(f"{symbol} 週K線{e}，可能是 Yahoo 的舊快取，{STALE_RETRY_WAIT_S} 秒後重新載入 ({attempt}/{STALE_RETRIES - 1})")
                    await asyncio.sleep(STALE_RETRY_WAIT_S)
            chart_points = await page.evaluate(_CHART_POINTS_JS)
        finally:
            await browser.close()
    return symbol, records, chart_points


def _to_number(s):
    return float(s.replace(",", ""))


def _check_against_chart(df, chart_points):
    """用圖表本身的 K 棒數值核對滑鼠讀到的開高低收，不一致就報錯"""
    if chart_points is None:
        print("(找不到頁面上的 Highcharts 物件，略過數值核對)")
        return
    taipei_tz = timezone(timedelta(hours=8))  # K 棒的時間戳記是台北時間當天 0 點
    chart = {}
    for x, *ohlc in chart_points:
        d = datetime.fromtimestamp(x / 1000, taipei_tz).date()
        chart[f"{d.year}/{d.month}/{d.day}"] = ohlc
    mismatched = [
        row["日期"] for _, row in df.iterrows()
        if row["日期"] not in chart
        or any(abs(_to_number(row[c]) - v) > 1e-6 for c, v in zip(["開", "高", "低", "收"], chart[row["日期"]]))
    ]
    if mismatched:
        raise RuntimeError(f"滑鼠讀到的數值與 K 線圖資料不一致，日期：{mismatched}")
    print(f"✔ {len(df)} 週的開高低收都與 K 線圖資料一致")


def fetch_stock_price_week(abbr, headless=True, timeout_s=30):
    """取得 K 線圖「週」的 日期/開/高/低/收，回傳 DataFrame (日期新到舊，格式比照其他週資料)"""
    symbol, records, chart_points = run_in_own_loop(
        _scrape_stock_price_week, abbr, headless=headless, timeout_s=timeout_s
    )
    df = pd.DataFrame(records, columns=PRICE_COLUMNS)
    for col in PRICE_COLUMNS[1:]:
        df[col] = df[col].str.replace(",", "")
    df = df.iloc[::-1].reset_index(drop=True)
    dates = [datetime.strptime(s, "%Y/%m/%d").date() for s in df["日期"]]
    df = _to_week_df(df.values.tolist(), dates, PRICE_COLUMNS, symbol)  # 日期改成 2026/10/8 (不補 0)
    _check_against_chart(df, chart_points)
    return df

In [4]:
# ===== 全部週資料（5 份 csv）：一次取得 =====

# 每份週資料的檔名標籤: 欄位 (檔名格式為 ABBR_{檔名標籤}_檔案建立年月日.csv)
WEEK_FILE_COLUMNS = {
    "Institutional_Trading_Week_8cols": INSTITUTIONAL_COLUMNS,
    "Margin_Week_11cols": MARGIN_COLUMNS,
    "Securities_Lending_and_Borrowing_Week_8cols": LENDING_COLUMNS,
    "Major_Holders_Week_5cols": MAJOR_HOLDERS_COLUMNS,
    PRICE_FILE_TAG: PRICE_COLUMNS,
}


def fetch_all_week_data(abbr, headless=True, timeout_s=30):
    """依序取得法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價 5 份週資料，回傳 {檔名標籤: DataFrame}
    (任何一份失敗就直接報錯，不會只回傳部分資料)"""
    df_inst = fetch_institutional_trading_week(abbr, headless=headless, timeout_s=timeout_s)
    df_margin, df_lending = fetch_margin_week(abbr, headless=headless, timeout_s=timeout_s)
    df_major = fetch_major_holders_week(abbr, headless=headless, timeout_s=timeout_s)
    df_price = fetch_stock_price_week(abbr, headless=headless, timeout_s=timeout_s)
    return {
        "Institutional_Trading_Week_8cols": df_inst,
        "Margin_Week_11cols": df_margin,
        "Securities_Lending_and_Borrowing_Week_8cols": df_lending,
        "Major_Holders_Week_5cols": df_major,
        PRICE_FILE_TAG: df_price,
    }

### 輸入公司代碼，開始蒐集資料

In [5]:
# ===== 執行：輸入代碼 → 取得「週」法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價 → 存檔 =====
ABBR = ask_stock_code()

# 想看瀏覽器操作過程可加 headless=False
week_data = fetch_all_week_data(ABBR)

# 5 份資料都抓成功後才清掉舊檔，讓 DataCollection/ABBR 資料夾內只保留最新的 5 份 csv
for old in clear_week_csv(ABBR, week_data):
    print(f"已刪除舊檔：{old.resolve()}")

for file_tag, df in week_data.items():
    csv_path = save_week_csv(df, ABBR, file_tag)
    print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path.resolve()}")
    display(df.head(3))

請輸入台股代碼(4位數字)：5498
✔ 120 週的開高低收都與 K 線圖資料一致
已刪除舊檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Institutional_Trading_Week_8cols_20261110.csv
已刪除舊檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Margin_Week_11cols_20261110.csv
已刪除舊檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Securities_Lending_and_Borrowing_Week_8cols_20261110.csv
已刪除舊檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Major_Holders_Week_5cols_20261110.csv
5498.TWO Institutional_Trading_Week_8cols 共 52 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Institutional_Trading_Week_8cols_20261110.csv


,日期,外資(張),投信(張),自營商(張),合計(張),外資籌碼,漲跌幅(%),成交量
0,2026/11/10,336,0,38,374,6.96%,6.63%,"5,299"
1,2026/11/6,86,0,-132,-45,6.73%,6.13%,"14,792"
2,2026/10/30,-53,0,-104,-158,6.53%,6.16%,"16,488"


5498.TWO Margin_Week_11cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Margin_Week_11cols_20261110.csv


,日期,融資,增減,餘額,使用率％,融券,增減,餘額,使用率％,券資比,資券互抵
0,2026/11/9,,37,"5,159",11.58%,,74,122,0.25%,2.36%,9
1,2026/11/2,,"-1,240","5,122",12.93%,,15,48,0.14%,0.94%,12
2,2026/10/26,,-733,"6,362",15.11%,,-48,33,0.11%,0.52%,28


5498.TWO Securities_Lending_and_Borrowing_Week_8cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Securities_Lending_and_Borrowing_Week_8cols_20261110.csv


,日期,融券,增減,餘額,限額,借券賣出,增減,餘額
0,2026/11/9,,74,122,-,,16,"6,923"
1,2026/11/2,,15,48,-,,141,"6,907"
2,2026/10/26,,-48,33,-,,-29,"6,766"


5498.TWO Major_Holders_Week_5cols 共 50 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Major_Holders_Week_5cols_20261110.csv


,年度/日期,外資籌碼,大戶籌碼,董監持股,股價
0,2026/11/6,6.73%,29.39%,13.05%,39.80
1,2026/10/30,6.53%,29.39%,13.05%,49.55
2,2026/10/22,6.39%,29.39%,13.05%,58.4


5498.TWO Stock_Price_Week 共 120 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\5498\5498_Stock_Price_Week_20261110.csv


,日期,開,高,低,收
0,2026/11/10,40.4,47.8,40.2,46.65
1,2026/11/6,47.5,51.5,38.2,39.8
2,2026/10/30,58.4,58.7,49.35,49.55
